<a href="https://colab.research.google.com/github/ksuaray/M4DS/blob/MATH-170-Spring-2026/Lab3_DE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MATH 170: Calculus for Data Science**


---



# **Lab 3: DE1 → DE2: One-Parameter Least Squares + Tangent Lines**

## What you will practice
### Least squares (DE1 idea)
- Fit a simple model **with one parameter**:
  $$\hat y = w x$$
- Build the least-squares error (loss):
  $$E(w)=\sum_{i=1}^n (y_i-w x_i)^2$$

### Derivatives + tangent lines (DE2 ideas)
- Compute $E'(w)$ with SymPy.
- Interpret $E'(w)$ as the **slope** of the loss graph at a chosen $w$.
- Write the **equation of the tangent line**:
  $$L(w)=E(w_0)+E'(w_0)(w-w_0)$$
- Do derivative practice with exponential, log, and trig functions, then plot each function with its tangent line.

We will use:
- **Pandas** for data
- **SymPy** for derivatives
- **Plotly Express** for graphs


---

## Part 0 — Setup

Run this cell first.


In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.express as px

sp.init_printing()

# Symbols for calculus
x = sp.Symbol('x', real=True)
w = sp.Symbol('w', real=True)


---

## Part 1 — Data and the one-parameter model  $\hat y = w x$

Let's create the dataset from DE1 and work through that example to remind ourselves how we developed the need for derivatives. The most important type of dataset you'll be creating as a data scientist using Python is called a **DataFrame**, and it is an object in the [Pandas](https://pandas.pydata.org/) library.


In [ ]:
# Data
x_data = np.array([1, 2, 3, 4], dtype=float)
y_data = np.array([8, 7, 13, 16], dtype=float)

df = pd.DataFrame({"x": x_data, "y": y_data})
df

Previously we made graphs with .... Today, we'll use **Plotly Express**, a powerful library for interactive graphics.

In [ ]:
# Plot the data
fig = px.scatter(df, x="x", y="y", title="Data points")
fig.show()


### 1.1 Try a few values of $w$ and see the line

For a chosen $w$, the model predicts: $\hat y = w x$.


In [ ]:
def predictions(w_val, x_vals):
    return w_val * x_vals

# Try a few w values
w_try = [2, 3, 4, 5, 6]

# Build lines for plotting
xs = np.linspace(df["x"].min() - 0.5, df["x"].max() + 0.5, 60)
fig = px.scatter(df, x="x", y="y", title="Data with a few candidate lines")

for w_val in w_try:
    fig = fig.add_scatter(x=xs, y=predictions(w_val, xs), mode="lines", name=f"w={w_val}")

fig.show()


**GTW:** Which $w$ looks *closest* by eye? Write your guess in the cell below:


My guess: $w=$

---

## Part 2 — Build the least squares loss $E(w)$ (one variable)

Error (loss) for point $i$:
$$l_i(w)=y_i-w x_i$$

Least squares loss:
$$E(w)=\sum_{i=1}^n (y_i-w x_i)^2$$


### 2.1 Construct $E(w)$ symbolically (SymPy)


In [ ]:
E = 0
for xi, yi in zip(x_data, y_data):
    E += (w*xi - yi)**2

E_s = sp.simplify(E)
E_s


### 2.2 Plot $E(w)$ as a curve

This is a 1D graph: input $w$ → output loss.


In [ ]:
E_num = sp.lambdify(w, E_s, "numpy")

w_vals = np.linspace(-1, 8, 300)
loss_vals = E_num(w_vals)

loss_df = pd.DataFrame({"w": w_vals, "E(w)": loss_vals})

fig = px.line(loss_df, x="w", y="E(w)", title="Loss curve E(w)")
fig.show()


**GTW:** On the graph of $E(w)$, where does the minimum appear to be (roughly)? State your answer below.


The minimum value seems to be $E(w)=$

It occurs at $w=$

---

## Part 3 — Derivative of the loss: $E'(w)$

It helps that we are able to visualize the loss function and create an expression for $E(w)$. This is doable in this case because we selected a relatively simple loss (squared distance) with a very simple model (only a slope parameter, $w$). But the models that power LLMs have *billions* of parameters, so we won't be able to visualize or write down their loss functions. *We need a gps that can tell us which direction to go to improve our parameter estimates.* That gps is known as *gradient descent*, and in this class, that is equivalent to using the derivative to make that decision.




### 3.2 Difference quotient check (derivative ≈ slope)

Approximate slope at $w_0$ using:

$$\frac{E(w_0+h)-E(w_0)}{h}.$$


In [ ]:
dE_num = sp.lambdify(w, dE, "numpy")

w0 = 3.0
h = 1e-4

approx_slope = (E_num(w0 + h) - E_num(w0)) / h
true_slope = dE_num(w0)

print("At w0 =", w0)
print("difference quotient slope =", approx_slope)
print("derivative E'(w0)         =", true_slope)


**GTW:** Change `w0` to 2, 4, and 5.
- When is the slope positive? negative?
- How does that tell you which direction moves you toward the minimum?


**GTW:** Change `w0` to 2, 4, and 5.
- When is the slope positive? negative?
- How does that tell you which direction moves you toward the minimum?


Now we compute the derivative:

$$E'(w)=\frac{d}{dw}E(w).$$

Meaning:
- $E'(w_0)$ is the **slope of the loss curve** at $w_0$.


### 3.1 Compute $E'(w)$ with SymPy


In [ ]:
dE = sp.simplify(sp.diff(E_s, w))
dE


**GTW:** In the code cell below, find $E(w_0)$ for $w_0$ = 2, 4, and 5.
- How do the values compare to what you calculated above?
- Can you use the expression for $E'(w)$ to determine when the slope is positive? negative? Write your answer in interval notation.



---

## Part 4 — Tangent line to the loss curve

For a function $E(w)$, the tangent line at $w_0$ is:


*Find an expression for the tangent line*


### 4.1 Compute and plot the tangent line on top of $E(w)$


In [ ]:
w0 = 3.0

E0 = float(E_s.subs(w, w0))
m0 = float(dE.subs(w, w0))

L = sp.simplify(E0 + m0*(w - w0))  # tangent line expression
L_num = sp.lambdify(w, L, "numpy")

loss_df["Tangent at w0"] = L_num(w_vals)

fig = px.line(loss_df, x="w", y=["E(w)", "Tangent at w0"], title=f"E(w) and tangent line at w0={w0}")
fig.show()

print("Tangent line L(w) =", L)


**GTW:** Change `w0` to a value near the minimum you saw in Part 2.
- Does the tangent line become almost flat there? Why?


---

## Part 5 — Find the best $w$ by solving $E'(w)=0$

In a 1D problem, *the minimum occurs where the slope is zero* (in nice cases):

$$E'(w)=0.$$


In [ ]:
w_star = sp.solve(sp.Eq(dE, 0), w)
w_star


In [ ]:
w_star_val = float(w_star[0])
print("Best-fit w =", w_star_val)
print("Loss at best-fit w =", float(E_s.subs(w, w_star_val)))

fig = px.line(loss_df, x="w", y="E(w)", title="Loss curve E(w) with best-fit w* marked")
fig = fig.add_scatter(x=[w_star_val], y=[E_num(w_star_val)], mode="markers", name="w*")
fig.show()


### 5.1 Plot the best-fit model line on the data


In [ ]:
xs = np.linspace(df["x"].min() - 0.5, df["x"].max() + 0.5, 60)
yhat_star = w_star_val * xs

fig = px.scatter(df, x="x", y="y", title=f"Best fit model: y = ({w_star_val:.3f}) x")
fig = fig.add_scatter(x=xs, y=yhat_star, mode="lines", name="best-fit line")
fig.show()


---

## Part 6 — Derivative practice + tangent lines (DE2 skill)

For each function:
1. Compute $f'(x)$ with SymPy
2. Choose a point $x_0$
3. Build the tangent line:
   Find an expression for the equation of a tangent line to a curve at a point $x_0*.
4. Plot $f$ and $L$



### Helper functions


In [ ]:
def tangent_line_expr(f_expr, x0):
    fp = sp.diff(f_expr, x)
    f0 = sp.simplify(f_expr.subs(x, x0))
    m0 = sp.simplify(fp.subs(x, x0))
    Lx = sp.simplify(f0 + m0*(x - x0))
    return fp, f0, m0, Lx

def plot_function_with_tangent(f_expr, x0, x_min, x_max, title=""):
    fp, f0, m0, Lx = tangent_line_expr(f_expr, x0)
    f_num = sp.lambdify(x, f_expr, "numpy")
    L_num = sp.lambdify(x, Lx, "numpy")

    xs = np.linspace(float(x_min), float(x_max), 400)
    dfp = pd.DataFrame({"x": xs, "f(x)": f_num(xs), "tangent": L_num(xs)})

    fig = px.line(dfp, x="x", y=["f(x)", "tangent"], title=title)
    fig = fig.add_scatter(x=[float(sp.N(x0))], y=[float(sp.N(f0))], mode="markers", name="(x0, f(x0))")
    fig.show()

    return fp, f0, m0, Lx


### Example A (Exponential): $f(x)=4e^{x}$ at $x_0=0$


In [ ]:
f = 4*sp.exp(x)
x0 = 0

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=-1, x_max=1,
    title="f(x)=4*exp(x) and tangent line at x0=0"
)

display(f)
display(fp)
#display("tangent line:", Lx)


### Example B (Log): $$f(x)=x+ \ln(x)$$ at $x_0=1$


In [ ]:
f = x+sp.log(x)
x0 = 1

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=0.2, x_max=2,
    title="f(x)=x+ln(x) and tangent line at x0=1"
)

display(f)
display(fp)


### Example C (Trig): $f(x)=\sin(x)$ at $x_0=\pi/4$


In [ ]:
f = sp.sin(x)
x0 = sp.pi/4

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=-1, x_max=3,
    title="f(x)=sin(x) and tangent line at x0=pi/4"
)

display(f)
display(fp)


---

## **GTW**
Complete the following

For each:
- Use SymPy to verify your answers for $f'(x)$
- Find the equation of the tangent line
- Plot $f$ and the tangent line


In [ ]:
# f(x) = cos(3x) at x0 = 0
f = sp.cos(3*x)
x0 = 0

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=-2, x_max=2,
    title="f(x) = cos(3x) tangent at x0=0"
)

display(fp)
display("tangent line:", Lx)


In [ ]:
# f(x) = x^3-5x+2cos(x) at x0 = 1 (domain x>0)
f = x**3-5*x+2*sp.cos(x)
x0 = 2

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=0.2, x_max=3,
    title="f(x) = x^3-5x+2cos(x) tangent at x0=1"
)

display(fp)




---



---



# **LAB 3 Homework**

Complete the following activities by 11:59pm tonight. Submit the url (link) to this notebook in Canvas once you complete these exercises. You may work collaboratively with your classmates, but each student will be expected to submit their own work.

### 1. What does $E'(w_0)$ tell you about the loss curve at $w_0$?


### 2. If $E'(w_0)>0$, should you increase or decrease $w$ to move toward the minimum?



### 3. Write the tangent line formula for a general function $f$ at $x_0$.


### 4. Go back to the beginning of this notebook and generate a different data set as below. Then run the code to create the line of best fit. Place that plot below the code for the data

In [ ]:
# Your Data. Note the new name for the DataFrame!!!

m = ... #Remove the ... and replace it with your birth month
n = ... #Remove the ... and replace it with the number of courses you are enrolled in this semester
d = ... #Remove the ... and replace it with the day of your birth
c = ... #Remove the ... and replace it with the number of units you are enrolled in this semester

x_data1 = np.array([1, 2, 3, 4, m, n], dtype=float)
y_data1 = np.array([8, 7, 13, 16, d, c], dtype=float)

df1 = pd.DataFrame({"x": x_data, "y": y_data})
df1

### 5. Plot the function $f(x)=tan x$ and it's tangent line at $x=\pi/6$.
Write the equation of the tangent line.

In [ ]:
# OPTION 3: f(x) = tan(x) at x0 = pi/6 (avoid undefined tan points)
f = ...
x0 = sp.pi/6

fp, f0, m0, Lx = plot_function_with_tangent(
    f, x0, x_min=-1, x_max=1,
    title="OPTION 3: tan(x) tangent at x0=pi/6"
)

display(fp)
display("tangent line:", Lx)
